<a href="https://colab.research.google.com/github/Eleonwra/elcardiocc-baseline-ner/blob/main/model_training/mbert_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Importing Dataset

In [2]:
%pip install datasets

In [3]:
import pickle
import datasets

In [4]:
with open('final_dataset_sentences.pickle', 'rb') as data:
    final_dataset_sentences = pickle.load(data)

##Checking lengths

In [5]:
from transformers import BertTokenizerFast
tokenizer = BertTokenizerFast.from_pretrained("bert-base-multilingual-cased")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
subset = ['train', 'validation']
tokenized_data = {'train': [], 'validation': []}
max_lengths = {name: 0 for name in subset}
sent_lengths = {'train': [], 'validation': []}
for section_name, section_data in final_dataset_sentences.items():
    for i in range(len(section_data)):
        texts =  ["".join(t) for t in section_data[i]["sentences_tokens"]]
        tokenized_document = tokenizer(texts)
        for j in range(len(tokenized_document['input_ids'])):
            tokenized_data[section_name].append(tokenized_document)
            sent_lengths[section_name].append(len(tokenized_document['input_ids'][j]))
            max_lengths[section_name] = max(max_lengths[section_name], len(tokenized_document['input_ids'][j]))
print("Max Length of sentences:", max_lengths)

Max Length of sentences: {'train': 458, 'validation': 442}


In [7]:
def count_elements_greater_than_threshold(values, threshold):
    return sum(1 for value in values if value > threshold)

for split in ['train', 'validation']:
    threshold = 384
    count_greater_than_threshold = count_elements_greater_than_threshold(sent_lengths[split], threshold)
    print(f"Number of elements greater than {threshold} in {split}: {count_greater_than_threshold}")

Number of elements greater than 384 in train: 4
Number of elements greater than 384 in validation: 3


##Tokenization

In [15]:
def tokenize_and_align_labels(data_section, max_length):
    texts = ["".join(t) for t in data_section["sentences_tokens"]]
    tokenized_inputs = tokenizer(texts, padding='max_length', truncation=True, max_length=max_length)
    labels = []
    for row_idx, label_old in enumerate(data_section["sentences_ner_tags"]):
        label_new = [[] for t in tokenized_inputs.tokens(batch_index=row_idx)]
        for char_idx in range(len(data_section["sentences_tokens"][row_idx])):
            token_idx = tokenized_inputs.char_to_token(row_idx, char_idx)
            if token_idx is not None:
                label_new[token_idx].append(data_section["sentences_ner_tags"][row_idx][char_idx])
        label_new = list(map(lambda i : max(i, default=0), label_new))
        labels.append(label_new)
    tokenized_inputs["labels"] = labels + [0] * (max_length - len(labels))
    return tokenized_inputs

In [16]:
section_name = ['train', 'validation']
tokenized_data = {'train': [], 'validation': []}
max_lengths = {
    'train': 384,
    'validation': 384
}

for section_name, section_data in final_dataset_sentences.items():
    for i in range(len(section_data)):
        tokenized_document = tokenize_and_align_labels(section_data[i], max_lengths[section_name])
        tokenized_data[section_name].append(tokenized_document)

##Save

In [18]:
with open('mBERT_tokenized_documents_trun.pkl', 'wb') as f:
    pickle.dump(tokenized_data, f)